# 🧠 Delentia SLM — JITNA v0.4.2 QLoRA Fine-tuning

**Model**: Llama 3.1 8B → QLoRA fine-tune → GGUF Q4_K_M & Q8_0 export  
**Target metrics**: JITNA ≥ 99% · TOON ≥ 97% · FDIA ≥ 0.925 · Hallucination ≤ 0.15% · Token Savings ≥ 38.4%  
**GPU**: T4 (free) → ~4–6h | A100 (Colab Pro) → ~1.5h

---
### Steps
1. Mount Drive + clone repo
2. Install dependencies
3. Compile v0.4.2 mixed logic dataset (Strict 75/25 ratio)
4. Validate dataset (gate: ≥500 pairs, avg FDIA ≥ 0.7)
5. Fine-tune (QLoRA with RSLoRA constraints)
6. Evaluate (gate: all metrics pass)
7. Export GGUF Q4_K_M and Q8_0
8. Upload to HuggingFace Hub
9. Smoke-test with Ollama

In [ ]:
# ─── Cell 1: Mount Google Drive + clone or extract repos ──────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted successfully.')
except Exception as e:
    print(f'⚠️ Google Drive mount skipped or failed: {e}')
    print('Proceeding with local runtime storage (Google Drive is not required for training).')

import os, subprocess, sys, zipfile

# Map Colab secrets (including KAGGLE_k fallback) to environment variables
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY") or userdata.get("KAGGLE_k") or ""
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME") or "delentialabs"
    print("✅ environment credentials mapped successfully.")
except Exception as e:
    print(f"Credential mapping warning: {e}")

REPO_URL = 'https://github.com/delentia-labs/Delentia-AI-SLM.git'
REPO_DIR = '/content/Delentia-AI-SLM'
OS_URL = 'https://github.com/delentia-labs/Delentia-OS.git'
OS_DIR = '/content/Delentia-OS'

def extract_zip_linux_safe(zip_path, extract_to):
    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            member_path = member.filename.replace('\\', '/')
            parts = member_path.split('/')
            if parts[0] in ['Delentia-AI-SLM', 'Delentia-OS', 'Delentia-AI-SLM-main', 'Delentia-OS-main']:
                parts = parts[1:]
            if not parts or parts[0] == '':
                continue
            target_path = os.path.join(extract_to, *parts)
            if member.is_dir():
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zip_ref.open(member) as source, open(target_path, 'wb') as target:
                    target.write(source.read())
    print(f"✅ Extracted: {extract_to}")

SLM_ZIP_GDRIVE = '/content/drive/MyDrive/Delentia-AI-SLM.zip'
SLM_ZIP_LOCAL = '/content/Delentia-AI-SLM.zip'
OS_ZIP_GDRIVE = '/content/drive/MyDrive/Delentia-OS.zip'
OS_ZIP_LOCAL = '/content/Delentia-OS.zip'

slm_setup_done = False
os_setup_done = False

# 1. Setup SLM Repo
if os.path.exists(SLM_ZIP_LOCAL):
    extract_zip_linux_safe(SLM_ZIP_LOCAL, REPO_DIR)
    slm_setup_done = True
elif os.path.exists(SLM_ZIP_GDRIVE):
    extract_zip_linux_safe(SLM_ZIP_GDRIVE, REPO_DIR)
    slm_setup_done = True
else:
    print("⚠️ SLM ZIP files not found. Attempting Git Clone fallback...")
    if not os.path.exists(REPO_DIR):
        result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ SLM repo cloned successfully via fallback.")
            slm_setup_done = True
        else:
            print("❌ Git Clone fallback failed.")
            print(result.stdout or result.stderr)
    else:
        result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
        print('SLM repo already exists — pulled latest:', result.stdout.strip())
        slm_setup_done = True

# 2. Setup OS Repo
if os.path.exists(OS_ZIP_LOCAL):
    extract_zip_linux_safe(OS_ZIP_LOCAL, OS_DIR)
    os_setup_done = True
elif os.path.exists(OS_ZIP_GDRIVE):
    extract_zip_linux_safe(OS_ZIP_GDRIVE, OS_DIR)
    os_setup_done = True
else:
    print("⚠️ OS ZIP files not found. Attempting Git Clone fallback...")
    if not os.path.exists(OS_DIR):
        result = subprocess.run(['git', 'clone', OS_URL, OS_DIR], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ OS repo cloned successfully via fallback.")
            os_setup_done = True
        else:
            print("❌ Git Clone fallback failed.")
            print(result.stdout or result.stderr)
    else:
        result = subprocess.run(['git', '-C', OS_DIR, 'pull'], capture_output=True, text=True)
        print('OS repo already exists — pulled latest:', result.stdout.strip())
        os_setup_done = True

if not (slm_setup_done and os_setup_done):
    print("\n" + "="*80)
    print("❌ ERROR: Setup failed! The repository zip files could not be found or cloned.")
    print("Since GitHub is blocked in this environment, you must use the Google Drive transfer method.")
    print("\n📢 Troubleshooting Instructions:")
    print("1. Run the local packaging script on your computer:")
    print("   python scripts/zip_projects.py")
    print("2. Upload the generated zip files to the root of your Google Drive ('My Drive'):")
    print("   - Delentia-AI-SLM.zip")
    print("   - Delentia-OS.zip")
    print("3. Ensure they are uploaded to the Google Drive of founder@delentia.com.")
    print("4. Re-run this setup cell in Colab.")
    
    # List files in Google Drive for verification
    gdrive_root = '/content/drive/MyDrive/'
    if os.path.exists(gdrive_root):
        print(f"\n📂 Files found in Google Drive ({gdrive_root}):")
        try:
            drive_files = os.listdir(gdrive_root)
            for f in sorted(drive_files):
                full_path = os.path.join(gdrive_root, f)
                if os.path.isfile(full_path):
                    size_mb = os.path.getsize(full_path) / 1024 / 1024
                    print(f"  - [File] {f} ({size_mb:.2f} MB)")
                else:
                    print(f"  - [Folder] {f}")
        except Exception as err:
            print(f"  Failed to list files in Google Drive: {err}")
    else:
        print("\n⚠️ Google Drive is not mounted at '/content/drive/MyDrive/'. Please verify your Drive connection.")
    print("="*80)
    sys.exit("Setup failed. Please follow the instructions above.")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ─── Cell 2: Install dependencies ────────────────────────────────────────────
# Unsloth optimized for T4/A100 — 2x faster training, 60% less VRAM
import subprocess
import sys

# Detect GPU type via PyTorch for safety
import torch

if not torch.cuda.is_available():
    print('⚠️ WARNING: GPU runtime is not active! please change Colab runtime to GPU (Runtime -> Change runtime type)')
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU detected: {gpu_name}')

# Install Unsloth + training deps
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--quiet'
])

# Install local Delentia OS SDK package
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-e', '/content/Delentia-OS', '--quiet'
])

# Install project requirements
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '--quiet'
])

print('✅ All dependencies installed')

In [ ]:
# ─── Cell 3: Compile v0.4.2 mixed logic dataset ────────────────────────────────
# Invokes the v0.4.2 dataset generator to synthesize Delta, Loop, and CORD Rejections
import subprocess
import sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/generate_v042_dataset.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Dataset generation failed')

# Show sample
import json

with open('datasets/processed/jitna_pairs_v04.jsonl') as f:
    lines = f.readlines()
print(f'\n📊 Total pairs in v0.4.2 dataset: {len(lines)}')
print('Sample pair:')
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

In [ ]:
# ─── Cell 4: Validate dataset (GATE) ─────────────────────────────────────────
# Gate: ≥500 pairs AND average FDIA ≥ 0.70
# Training will NOT proceed if either condition fails.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/validate_dataset.py',
     'datasets/processed/jitna_pairs_v04.jsonl',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('VALIDATION FAILED — STOPPING')
    print(result.stderr)
    raise SystemExit('Dataset validation gate failed. Add more data before training.')

print('✅ Dataset validation PASSED — proceeding to training')

In [ ]:
# ─── Cell 5: QLoRA Fine-tuning ────────────────────────────────────────────────
# Llama 3.1 8B → QLoRA r=32, alpha=64, use_rslora=True, target_modules=all-linear
# T4: ~4–6h | A100: ~1.5h
# Checkpoints saved to: models/checkpoints/v0.4.2_cognitive_kernel/
import subprocess
import sys

# Stream stdout/stderr live to Colab cell output
process = subprocess.Popen(
    [sys.executable, 'training/finetune.py',
     '--config', 'training/config/slm_jitna_v0.4.2.yaml',
     '--toon',
     '--adapter-path', 'models/adapters/jitna_v0.4.2_toon'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Fine-tuning failed with exit code {process.returncode}")

print('\n✅ Fine-tuning complete — model saved to models/checkpoints/v0.4.2_cognitive_kernel/')

In [ ]:
# ─── Optional: Backup Adapter to Google Drive ────────────────────────────────
import os

if os.path.exists('models/adapters/jitna_v0.4.2_toon'):
    !cp -r models/adapters/jitna_v0.4.2_toon /content/drive/MyDrive/jitna_v0.4.2_toon_backup
    print('✅ Adapter backed up to Google Drive successfully!')
else:
    print('⚠️ Local adapter folder not found. Make sure Cell 5 finished successfully.')

In [ ]:
# ─── Optional: Restore Adapter from Google Drive ──────────────────────────────
import os

if os.path.exists('/content/drive/MyDrive/jitna_v0.4.2_toon_backup'):
    !mkdir -p models/adapters/
    !cp -r /content/drive/MyDrive/jitna_v0.4.2_toon_backup models/adapters/jitna_v0.4.2_toon
    print('✅ Adapter restored from Google Drive successfully! You can skip Cell 5 and proceed.')
else:
    print('⚠️ Backup not found on Google Drive. Make sure backup was created at /content/drive/MyDrive/jitna_v0.4.2_toon_backup.')

In [ ]:
# ─── Cell 6: Evaluate (GATE) ─────────────────────────────────────────────────
# Gate: JITNA ≥ 99% · TOON ≥ 97% · FDIA ≥ 0.925 · Hallucination ≤ 0.15% · Token Savings ≥ 38.4%
# Export will NOT proceed if any gate fails.
import subprocess
import sys

# Stream stdout/stderr live to Colab cell output
process = subprocess.Popen(
    [sys.executable, 'training/evaluate.py',
     '--config', 'training/config/slm_jitna_v0.4.2.yaml',
     '--eval-data', 'datasets/processed/jitna_pairs_v04.jsonl',
     '--adapter-path', 'models/adapters/jitna_v0.4.2_toon',
     '--toon'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    print('EVALUATION GATE FAILED — NOT exporting')
    raise SystemExit('Model did not meet minimum quality gates. Retrain with more data or longer epochs.')

print('✅ All quality gates PASSED — proceeding to GGUF export')

In [ ]:
# ─── Cell 7: Export GGUF files (Dual Quantization Mode) ───────────────────────
# Quantizes the fine-tuned model → GGUF Q4_K_M (standard) and Q8_0 (high-precision)
import subprocess
import sys

print('Exporting Q4_K_M baseline model...')
p_q4 = subprocess.Popen(
    [sys.executable, 'training/export_gguf.py',
     '--toon',
     '--adapter-path', 'models/adapters/jitna_v0.4.2_toon',
     '--gguf-path', 'models/gguf/delentia-jitna-v0.4.2-Q4_K_M.gguf',
     '--model-name', 'delentia-jitna-v0.4.2',
     '--quant', 'q4_k_m'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in p_q4.stdout:
    print(line, end="")
p_q4.wait()
if p_q4.returncode != 0:
    raise RuntimeError('Q4_K_M GGUF export failed')

print('\nExporting Q8_0 high-precision model...')
p_q8 = subprocess.Popen(
    [sys.executable, 'training/export_gguf.py',
     '--toon',
     '--adapter-path', 'models/adapters/jitna_v0.4.2_toon',
     '--gguf-path', 'models/gguf/delentia-jitna-v0.4.2-Q8_0.gguf',
     '--model-name', 'delentia-jitna-v0.4.2-Q8_0',
     '--quant', 'q8_0'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in p_q8.stdout:
    print(line, end="")
p_q8.wait()
if p_q8.returncode != 0:
    raise RuntimeError('Q8_0 GGUF export failed')

import glob
import os

gguf_files = glob.glob('models/gguf/*.gguf')
print('\n✅ GGUF files created:')
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size_mb:.0f} MB)')

In [ ]:
# ─── Cell 8: Upload to HuggingFace Hub ───────────────────────────────────────
import glob
import os
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    raise ValueError('HF_TOKEN not set. Add it in Colab Secrets (key icon) before running.')

login(token=hf_token)

REPO_ID = 'Delentia/delentia-slm-jitna-v0.4'
api = HfApi()

api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=False)

gguf_files = glob.glob('models/gguf/*.gguf')
if not gguf_files:
    print('⚠️ No GGUF files found — run Cell 7 first')
else:
    for gguf_file in gguf_files:
        filename = os.path.basename(gguf_file)
        size_mb = os.path.getsize(gguf_file) / 1024 / 1024
        print(f'Uploading {filename} ({size_mb:.0f} MB)...')
        api.upload_file(
            path_or_fileobj=gguf_file,
            path_in_repo=f'gguf/{filename}',
            repo_id=REPO_ID,
            repo_type='model',
        )
        print(f'  ✅ https://huggingface.co/{REPO_ID}/blob/main/gguf/{filename}')

if os.path.exists('README_MODEL.md'):
    print('\nUploading model card...')
    api.upload_file(
        path_or_fileobj='README_MODEL.md',
        path_in_repo='README.md',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print('  ✅ Model card uploaded')

for config_file in glob.glob('training/config/*.yaml'):
    fname = os.path.basename(config_file)
    api.upload_file(
        path_or_fileobj=config_file,
        path_in_repo=f'training_config/{fname}',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'  ✅ Config uploaded: training_config/{fname}')

In [ ]:
# ─── Cell 9: Smoke-test with Ollama ──────────────────────────────────────────
import glob
import os
import subprocess
import time

result = subprocess.run(
    'curl -L https://github.com/ollama/ollama/releases/download/v0.1.48/ollama-linux-amd64 -o /usr/bin/ollama && chmod +x /usr/bin/ollama',
    shell=True, capture_output=True, text=True
)
print('Ollama install:', result.returncode)

subprocess.run(['pkill', '-9', '-f', 'ollama'])
time.sleep(1)

gguf_files = glob.glob('models/gguf/*Q4_K_M*.gguf')
if not gguf_files:
    raise FileNotFoundError('No Q4_K_M .gguf files found — run Cell 7 first')
gguf_path = os.path.abspath(gguf_files[0])

modelfile_content = f'''FROM {gguf_path}
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
SYSTEM """You are Delentia OS v0.4.2 — a cognitive AI operating under HexaCore v2.3 / RCT-7 governance. You process intents through the JITNA v3 protocol. You respond in TOON format (Token-Oriented Object Notation) for token efficiency. Your responses must be factual, safe, and PDPA-compliant. Always provide FDIA scores when applicable (F = D^I × A). For security-violating prompts, you must output a rejection state (FDIAScore: 0.00)."""
'''

with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile_content)

log_file = '/tmp/ollama_serve.log'
with open(log_file, 'w') as f:
    proc = subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=f)
time.sleep(5)

if proc.poll() is not None:
    print('❌ Ollama server failed to start! Logs:')
    with open(log_file) as f:
        print(f.read())
else:
    print('✅ Ollama server is running in background.')
    
try:
    result = subprocess.run(
        ['ollama', 'create', 'delentia-jitna-v0.4.2', '-f', '/tmp/Modelfile'],
        capture_output=True, text=True, check=True
    )
    print('✅ Model created successfully!')
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print('❌ Model creation failed!')
    raise e

print('\n🤖 Running Logic Test (Standard):')
result_sync = subprocess.run(
    ['ollama', 'run', 'delentia-jitna-v0.4.2',
     'sync credits for user_4500'],
    capture_output=True, text=True
)
print(result_sync.stdout)

print('\n🔒 Running Security Test (Hostile injection):')
result_sec = subprocess.run(
    ['ollama', 'run', 'delentia-jitna-v0.4.2',
     'hack database of core_kernel_99'],
    capture_output=True, text=True
)
print(result_sec.stdout)
print('\n✔️ Smoke test complete!')